In [1]:
pip install requests beautifulsoup4 pandas

     -------------------------------------- 11.4/11.4 MB 843.9 kB/s eta 0:00:00
     ---------------------------------------- 15.9/15.9 MB 1.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\faisal\envWebmining39\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Header agar request tidak diblokir oleh server detik
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}


def get_isi_berita(url_berita):
  """Fungsi tambahan untuk mengambil teks lengkap isi berita dari halaman detail"""
  try:
    resp = requests.get(url_berita, headers=HEADERS, timeout=10)
    if resp.status_code == 200:
      soup = BeautifulSoup(resp.text, 'html.parser')

      # Elemen utama penampung teks berita di detik.com
      konten = soup.select_one('div.detail__body-text') or soup.select_one(
          'article'
      )

      if konten:
        # Hapus elemen pengganggu seperti widget "Baca Juga", iklan, atau tag
        for unwanted in konten.select(
            'table, .baca-juga, .detail__body-tag, script, iframe'
        ):
          unwanted.decompose()

        # Ambil semua teks dari tag <p>
        paragraf = [p.get_text(strip=True) for p in konten.select('p')]
        isi_lengkap = ' '.join(paragraf)
        return isi_lengkap
  except Exception as e:
    print(f'Error saat mengambil isi berita dari {url_berita}: {e}')

  return ''


def crawl_detik(base_url, category_name, target_count=100):
  articles = []
  page = 1

  print(f'=== Mulai crawling {category_name} (Target: {target_count}) ===')

  while len(articles) < target_count:
    url = f'{base_url}?page={page}'
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
      print(
          f'Gagal mengakses halaman {page}. Status code:'
          f' {response.status_code}'
      )
      break

    soup = BeautifulSoup(response.text, 'html.parser')
    items = soup.find_all('article')

    if not items:
      print(
          f'Tidak ada artikel lagi ditemukan di halaman {page}. Berhenti.'
      )
      break

    for item in items:
      if len(articles) >= target_count:
        break

      title_tag = item.find('h3', class_='media__title') or item.find('h2')
      link_tag = item.find('a')

      if title_tag and link_tag:
        title = title_tag.get_text(strip=True)
        link = link_tag.get('href', '')

        if link and 'detik.com' in link:
          # --- TAMBAHAN: Ambil isi berita lengkap ---
          print(f'Mengambil isi berita ({len(articles)+1}/{target_count}): {title[:40]}...')
          isi_berita = get_isi_berita(link)

          # Hanya masukkan jika isi berita berhasil terambil
          if isi_berita:
            articles.append({
                'kategori': category_name,
                'judul': title,
                'isi_berita': isi_berita,  # Kolom isi berita baru
                'url': link,
            })

          time.sleep(0.5)  # Jeda sebentar antar artikel

    print(
        f'Halaman {page} selesai | Berhasil mendapatkan {len(articles)}/{target_count} berita'
    )
    page += 1
    time.sleep(1)

  return articles


# --- Eksekusi Crawling ---

# 1. Crawling 100 Berita Sport
url_sport = 'https://sport.detik.com/indeks'
data_sport = crawl_detik(url_sport, 'Sport', target_count=100)

# 2. Crawling 100 Berita Finance
url_finance = 'https://finance.detik.com/indeks'
data_finance = crawl_detik(url_finance, 'Finance', target_count=100)

# 3. Gabungkan dan Simpan ke CSV
all_data = data_sport + data_finance
df = pd.DataFrame(all_data)

# Reorder kolom agar rapi: kategori, judul, isi_berita, url
df = df[['kategori', 'judul', 'isi_berita', 'url']]

# Simpan ke file CSV
output_filename = 'berita_detik_sport_finance.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(
    f'\nProses Selesai! Total {len(df)} berita berhasil disimpan ke'
    f' {output_filename}'
)

=== Mulai crawling Sport (Target: 100) ===
Mengambil isi berita (1/100): Ketum KONI Tutup Program Sport Diplomacy...
Mengambil isi berita (2/100): Usai Dilantik KONI, Ini Rencana Pengurus...
Mengambil isi berita (3/100): Lina Hisage, Atlet Muda Potensial Cabor ...
Mengambil isi berita (4/100): Tim Equestrian Indonesia Turunkan 12 Atl...
Mengambil isi berita (5/100): Serunya 549 Siswa di Ciamis Adu Ketangka...
Mengambil isi berita (6/100): 'Marc Marquez adalah Makhluk Mars'...
Mengambil isi berita (7/100): Alwi, Ubed, hingga Yusuf Petik Pengalama...
Mengambil isi berita (8/100): Viral soal Cepirit, Atlet Hyrox Ini Mala...
Mengambil isi berita (9/100): Dukung Eks Partner di Asian Games, Dejan...
Mengambil isi berita (10/100): Bagas/Apri Tak Minder Mulai dari Nol di ...
Mengambil isi berita (11/100): Alfamart Run 2026 Siap Digelar, 3,3 Juta...
Mengambil isi berita (12/100): Tatap Tur Eropa, Dejan/Felisha Lapar Gel...
Mengambil isi berita (13/100): Senna Agius Lengkapi Grid MotoGP 2027, S.

In [9]:
import pandas as pd

# 1. Atur batas tampilan pandas agar menampilkan semua baris dan teks penuh
pd.set_option('display.max_rows', None)  # Menampilkan seluruh 200 baris
pd.set_option(
    'display.max_colwidth', None
)  # Menampilkan seluruh isi berita tanpa terpotong

# 2. Baca file CSV
df = pd.read_csv('berita_detik_sport_finance.csv')

# 3. Tampilkan seluruh data
df

kategori  \
0      Sport   
1      Sport   
2      Sport   
3      Sport   
4      Sport   
5      Sport   
6      Sport   
7      Sport   
8      Sport   
9      Sport   
10     Sport   
11     Sport   
12     Sport   
13     Sport   
14     Sport   
15     Sport   
16     Sport   
17     Sport   
18     Sport   
19     Sport   
20     Sport   
21     Sport   
22     Sport   
23     Sport   
24     Sport   
25     Sport   
26     Sport   
27     Sport   
28     Sport   
29     Sport   
30     Sport   
31     Sport   
32     Sport   
33     Sport   
34     Sport   
35     Sport   
36     Sport   
37     Sport   
38     Sport   
39     Sport   
40     Sport   
41     Sport   
42     Sport   
43     Sport   
44     Sport   
45     Sport   
46     Sport   
47     Sport   
48     Sport   
49     Sport   
50     Sport   
51     Sport   
52     Sport   
53     Sport   
54     Sport   
55     Sport   
56     Sport   
57     Sport   
58     Sport   
59     Sport   
60     Sport   
61     Sport   
62     Sport   
63     Sport   
64     Sport   
65     Sport   
66     Sport   
67     Sport   
68     Sport   
69     Sport   
70     Sport   
71     Sport   
72     Sport   
73     Sport   
74     Sport   
75     Sport   
76     Sport   
77     Sport   
78     Sport   
79     Sport   
80     Sport   
81     Sport   
82     Sport   
83     Sport   
84     Sport   
85     Sport   
86     Sport   
87     Sport   
88     Sport   
89     Sport   
90     Sport   
91     Sport   
92     Sport   
93     Sport   
94     Sport   
95     Sport   
96     Sport   
97     Sport   
98     Sport   
99     Sport   
100  Finance   
101  Finance   
102  Finance   
103  Finance   
104  Finance   
105  Finance   
106  Finance   
107  Finance   
108  Finance   
109  Finance   
110  Finance   
111  Finance   
112  Finance   
113  Finance   
114  Finance   
115  Finance   
116  Finance   
117  Finance   
118  Finance   
119  Finance   
120  Finance   
121  Finance   
122  Finance   
123  Finance   
124  Finance   
125  Finance   
126  Finance   
127  Finance   
128  Finance   
129  Finance   
130  Finance   
131  Finance   
132  Finance   
133  Finance   
134  Finance   
135  Finance   
136  Finance   
137  Finance   
138  Finance   
139  Finance   
140  Finance   
141  Finance   
142  Finance   
143  Finance   
144  Finance   
145  Finance   
146  Finance   
147  Finance   
148  Finance   
149  Finance   
150  Finance   
151  Finance   
152  Finance   
153  Finance   
154  Finance   
155  Finance   
156  Finance   
157  Finance   
158  Finance   
159  Finance   
160  Finance   
161  Finance   
162  Finance   
163  Finance   
164  Finance   
165  Finance   
166  Finance   
167  Finance   
168  Finance   
169  Finance   
170  Finance   
171  Finance   
172  Finance   
173  Finance   
174  Finance   
175  Finance   
176  Finance   
177  Finance   
178  Finance   
179  Finance   
180  Finance   
181  Finance   
182  Finance   
183  Finance   
184  Finance   
185  Finance   
186  Finance   
187  Finance   
188  Finance   
189  Finance   
190  Finance   
191  Finance   
192  Finance   
193  Finance   
194  Finance   
195  Finance   
196  Finance   
197  Finance   
198  Finance   
199  Finance   

                                                                                          judul  \
0                                Ketum KONI Tutup Program Sport Diplomacy Indonesia-Timor Leste   
1                          Usai Dilantik KONI, Ini Rencana Pengurus Pusat Kickboxing Indonesia!   
2                               Lina Hisage, Atlet Muda Potensial Cabor Tolak Peluru dari Papua   
3                                Tim Equestrian Indonesia Turunkan 12 Atlet di Asian Games 2026   
4                        Serunya 549 Siswa di Ciamis Adu Ketangkasan Lomba Olahraga Tradisional   
5                                                            'Marc Marquez adalah Makhluk Mars'   
6                                   Alwi, Ubed, hingga Yusuf Petik Pengalaman dari Kento Momot